In [17]:
import os 
from dotenv import load_dotenv
from langchain_groq import ChatGroq 
from langchain_tavily import TavilySearch

load_dotenv()

groq_api_key=os.getenv("GROQ_API_KEY")
os.getenv('TAVILY_API_KEY')
model=ChatGroq(model='openai/gpt-oss-20b',groq_api_key=groq_api_key)
model


ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.7', 'langchain': '1.4.3'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001714BA4D810>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001714BA8C690>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [18]:
from langgraph.graph import StateGraph,START,END 
from langgraph.prebuilt import tool_node
from langgraph.prebuilt import tools_condition
from typing_extensions import TypedDict
from typing import Annotated
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

In [19]:
tavily_tool=TavilySearch()
tavily_tool.invoke('what is ai')


{'query': 'what is ai',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://en.wikipedia.org/wiki/Artificial_intelligence',
   'title': 'Artificial intelligence',
   'content': '"AI" redirects here. For other uses, see AI (disambiguation) "AI (disambiguation)") and Artificial intelligence (disambiguation) "Artificial intelligence (disambiguation)").\n\nPart of a series on\nArtificial intelligence (AI)\nshow\n\nMajor goals\n\n   Artificial general intelligence\n       superintelligence [...] The regulation of artificial intelligence is the development of public sector policies and laws for promoting and regulating AI; it is therefore related to the broader regulation of algorithms.( The regulatory and policy landscape for AI is an emerging issue in jurisdictions globally.( According to AI Index at Stanford, the annual number of AI-related laws passed in the 127 survey countries jumped from one passed in 2016 to 37 passed in 2022 alone.( Between 20

In [20]:
def calculator(expression: str) -> str:
    """
    Calculate a mathematical expression.

    Example:
        100 * 25
        500 / 10
        (100 + 50) * 2
    """

    try:

        allowed_characters = "0123456789+-*/().% "

        if not all(
            char in allowed_characters
            for char in expression
        ):
            return "Invalid mathematical expression."

        result = eval(
            expression,
            {"__builtins__": {}},
            {}
        )

        return str(result)

    except Exception as e:

        return f"Calculation error: {str(e)}"

In [21]:
def multiply(a: int, b: int) -> int:
    """
    Multiply two numbers.
    """

    return a * b

In [22]:
def percentage_change(
    old_value: float,
    new_value: float
) -> float:
    """
    Calculate percentage change.

    Formula:
        ((new_value - old_value) / old_value) * 100
    """

    if old_value == 0:

        return 0.0

    result = (
        (new_value - old_value)
        / old_value
    ) * 100

    return round(result, 2)

In [23]:
def analyze_text(text: str) -> dict:
    """
    Analyze basic statistics of a text.
    """

    words = text.split()

    sentences = [
        sentence
        for sentence in text.split(".")
        if sentence.strip()
    ]

    return {
        "characters": len(text),
        "words": len(words),
        "sentences": len(sentences)
    }

In [24]:
def calculate_average(
    numbers: list[float]
) -> float:
    """
    Calculate the average of a list of numbers.
    """

    if not numbers:

        return 0.0

    return round(
        sum(numbers) / len(numbers),
        2
    )

In [25]:
def find_maximum(
    numbers: list[float]
) -> float:
    """
    Find the maximum value.
    """

    if not numbers:

        return 0.0

    return max(numbers)

In [26]:
def find_minimum(
    numbers: list[float]
) -> float:
    """
    Find the minimum value.
    """

    if not numbers:

        return 0.0

    return min(numbers)

In [27]:
tools=[tavily_tool,calculator,multiply,percentage_change,analyze_text,calculate_average,find_maximum,find_minimum]

In [28]:
tools_list=model.bind_tools(tools)

In [29]:
tools_list

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.7', 'langchain': '1.4.3'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001714BA4D810>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001714BA8C690>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'tavily_search', 'description': 'A search engine optimized for comprehensive, accurate, and 

In [30]:
class State(TypedDict):
    messages:Annotated[list,add_messages]

graph_response=StateGraph(State)

In [32]:
def research_agent(state:State):
    response=tools_list.invoke(state['messages'])
    return{'messages':response}

builder=StateGraph(State)


builder.add_node("research_agent", research_agent)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "research_agent")

builder.add_conditional_edges(
    "research_agent",
    tools_condition
)

builder.add_edge("tools", "research_agent")

graph = builder.compile()

In [34]:
from langchain.messages import HumanMessage

response=graph.invoke({
    'messages':[
        HumanMessage(content='what is 2+2')
    ]
})

response

{'messages': [HumanMessage(content='what is 2+2', additional_kwargs={}, response_metadata={}, id='858ba893-9ad0-43da-971b-01c41713cc57'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Need to compute 2+2. Use calculator.', 'tool_calls': [{'id': 'fc_7886bf8f-e0f0-4dd2-931b-651bb32a6f40', 'function': {'arguments': '{"expression":"2+2"}', 'name': 'calculator'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 37, 'prompt_tokens': 491, 'total_tokens': 528, 'completion_time': 0.039035461, 'completion_tokens_details': {'reasoning_tokens': 12}, 'prompt_time': 0.029178262, 'prompt_tokens_details': None, 'queue_time': 0.395350622, 'total_time': 0.068213723}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_bb6a356524', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a15cb7-282c-7f03-bfd0-af75a6340bda-0', tool_calls=[{'name': 'calculator', 'args': {'expression': '2